# Assignment 4 - Field/soil mapping with SSURGO

## Purpose

This notebook maps the 25-field Story County sample produced in Assignment 2
against the official USDA NRCS SSURGO soil survey for Story County (SSA IA169)
and commits one reproducible map to `docs/assets/`. Every number below is
computed from the committed Assignment 4 products; no synthetic data is used.

## Sources

- Fields: USDA ACPF **Iowa Field Boundaries 2019**
  (`data/processed/assignment-02/fields_EPSG4326.geojson`), the same 25 fields
  selected in Assignment 2.
- Soils: USDA NRCS **SSURGO snapshot dated 2025-09-09** for soil survey area
  IA169 (`wss_SSA_IA169_[2025-09-09].zip`), reduced to the map units that
  intersect the 25 fields in `data/processed/assignment-04/soil_map_units.geojson`.
- Overlap: `data/processed/assignment-04/field_soil_overlap.csv`, one row per
  field/mapunit intersection with overlap area and per-field fraction computed
  in EPSG:5070.

## Snapshot and scale limitations

The soil layer is a fixed **2025-09-09** snapshot of SSURGO. SSURGO maps are
designed for interpretations at scales from about **1:12,000 to 1:63,360** and
are not intended for site-specific or larger-scale interpretations. Soil
polygon boundaries and the 2019 ACPF field boundaries have different vintages,
so small edge slivers or boundary offsets between the two layers are expected
artifacts of the two sources rather than evidence of a change on the ground.


## Load Assignment 2 fields and Assignment 4 soil products


In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# nbconvert starts the kernel in the notebook directory; locate the repo root
# by walking upward to the committed Assignment 4 products.
ROOT = Path.cwd()
for parent in (ROOT, *ROOT.parents):
    if (parent / "data" / "processed" / "assignment-04").is_dir():
        ROOT = parent
        break

ASSETS = ROOT / "docs" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

FIELDS_PATH = ROOT / "data/processed/assignment-02/fields_EPSG4326.geojson"
SOILS_PATH = ROOT / "data/processed/assignment-04/soil_map_units.geojson"
OVERLAP_PATH = ROOT / "data/processed/assignment-04/field_soil_overlap.csv"

fields = gpd.read_file(FIELDS_PATH)
soils = gpd.read_file(SOILS_PATH)
overlap = pd.read_csv(OVERLAP_PATH, dtype={"mukey": str})

assert len(fields) == 25 and fields["field_id"].nunique() == 25
assert soils["mukey"].is_unique and soils.crs.to_epsg() == 4326
assert fields.crs.to_epsg() == 4326
assert set(overlap["field_id"]) == set(fields["field_id"])
assert overlap["field_fraction"].between(0.0, 1.0).all()

print(f"fields: {len(fields)} features (EPSG:{fields.crs.to_epsg()})")
print(f"soil map units: {len(soils)} unique mukey (EPSG:{soils.crs.to_epsg()})")
print(f"overlap rows: {len(overlap)} across "
      f"{overlap['field_id'].nunique()} fields")


fields: 25 features (EPSG:4326)
soil map units: 47 unique mukey (EPSG:4326)
overlap rows: 165 across 25 fields


## CRS alignment

- The Assignment 2 field file and the SSURGO source shapefile are both
  delivered in **EPSG:4326** (WGS 84 latitude/longitude, degrees).
- Area calculations in degrees are not equal-area, so the overlap pipeline
  reprojects both layers to **EPSG:5070** (CONUS Albers Equal Area, metres)
  before computing intersection areas. Each field's fraction is its overlap
  area divided by its whole-field area in that equal-area projection, so
  partial coverage stays visible instead of being normalized to 1.0.
- The committed GeoJSON is exported back to **EPSG:4326** for interoperability
  with the rest of the pipeline; the map below is plotted in **EPSG:5070** so
  the axes are in metres and shapes are undistorted.


## Soil coverage report


In [2]:
coverage = overlap.groupby("field_id")["field_fraction"].sum()
gaps = coverage.loc[coverage < 1.0 - 1e-9]

print("per-field soil coverage fraction (sum of field_fraction):")
print(coverage.describe().round(6).to_string())
print(f"\nfields with explicit coverage gaps: {len(gaps)}")
if len(gaps):
    print(gaps.round(6).to_string())

mapped = overlap.loc[overlap["mukey"] != ""]
print(f"\noverlap rows: {len(overlap)}; map units covering the sample: "
      f"{mapped['mukey'].nunique()}")


per-field soil coverage fraction (sum of field_fraction):
count    25.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0

fields with explicit coverage gaps: 0

overlap rows: 165; map units covering the sample: 47


## Map: selected fields over soil map units


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

field_work = fields.to_crs(5070)
soil_work = soils.to_crs(5070).copy()
color_index = pd.factorize(soil_work["musym"])[0] % 20
cmap = plt.get_cmap("tab20")

fig, ax = plt.subplots(figsize=(10, 8))
soil_work.plot(
    ax=ax,
    color=[cmap(index) for index in color_index],
    alpha=0.55,
    edgecolor="#444444",
    linewidth=0.3,
)
field_work.boundary.plot(ax=ax, color="black", linewidth=1.4)
ax.set_title("Selected Story County fields over SSURGO soil map units")
ax.set_xlabel("Easting (m, EPSG:5070)")
ax.set_ylabel("Northing (m, EPSG:5070)")
fig.tight_layout()
fig.text(
    0.01, 0.01,
    "Source: USDA NRCS SSURGO snapshot 2025-09-09, soil survey area IA169; "
    "field outlines from USDA ACPF Iowa Field Boundaries 2019 (n = 25). "
    "Plot CRS: EPSG:5070 (metres).",
    fontsize=8,
)
fig.savefig(ASSETS / "field_spatial_map.png", dpi=160)
plt.close(fig)
print("wrote docs/assets/field_spatial_map.png")


wrote docs/assets/field_spatial_map.png


## Most common mapunit by field-overlap area


In [4]:
by_area = (
    overlap.loc[overlap["mukey"] != ""]
    .groupby(["mukey", "musym", "muname"], as_index=False)["overlap_area_m2"]
    .sum()
    .sort_values("overlap_area_m2", ascending=False)
)
total_overlap_ha = by_area["overlap_area_m2"].sum() / 10000.0
top = by_area.iloc[0]
top_ha = top["overlap_area_m2"] / 10000.0

print("top 3 map units by summed field-overlap area:")
print(
    by_area.head(3)
    .assign(overlap_area_ha=lambda frame: frame["overlap_area_m2"] / 10000.0)[
        ["mukey", "musym", "muname", "overlap_area_ha"]
    ]
    .round(3)
    .to_string(index=False)
)
print(
    f"\nmost common mapunit by field-overlap area: {top['musym']} "
    f"({top['muname']}), mukey {top['mukey']}, {top_ha:.2f} ha "
    f"({top_ha / total_overlap_ha * 100.0:.1f}% of mapped overlap)"
)


top 3 map units by summed field-overlap area:
  mukey musym                                                  muname  overlap_area_ha
2765537 L138B      Clarion loam, Bemis moraine, 2 to 6 percent slopes          193.956
2835021  L107 Webster clay loam, Bemis moraine, 0 to 2 percent slopes          150.331
2800480   L55                    Nicollet loam, 1 to 3 percent slopes          112.086

most common mapunit by field-overlap area: L138B (Clarion loam, Bemis moraine, 2 to 6 percent slopes), mukey 2765537, 193.96 ha (24.6% of mapped overlap)


## Interpretation and limitations

- The coverage report above shows the per-field soil coverage fraction; any
  field whose polygons leave part of the field unmapped is listed in the
  explicit gap report with fractions summing below 1.0.
- The most common mapunit above is ranked by **summed field-overlap area** in
  EPSG:5070, not by polygon count.
- All soil attributes come from the fixed **2025-09-09 IA169 snapshot**; the
  map does not reflect any survey revisions after that date, and SSURGO is not
  designed for site-specific or larger-scale interpretations (about 1:12,000
  to 1:63,360). The field outlines are 2019 ACPF polygons and do not represent
  current ownership or program boundaries.
